# Intermediate-grid supervision (SFT) for Aligned AR — 4 slots, trained k = 0, 2 (seed 42)

**Why.** Aligned AR fails on this setting (notebook 42): it applies a single operation twice (A → AA 100% at
slots 1–3) and drops the middle operation of a 3-operation instruction. Our explanation is an ordinal template
("1st … last" roles) that fits every final answer. Supervising the grid after EVERY slot rules that template out:
the model must execute slot by slot, and `<identity>` must leave the grid unchanged.

**Model.** Aligned AR (no sequence position, op-slot embedding, target-cell coordinates, 10-way digit head), but the
output is the trace:

`<BOS> <OPS> [4 slots] <GRID> [36 input] <OUTPUT> [grid after slot 1] [after slot 2] [after slot 3] [after slot 4] <EOS>`

The last grid is the answer, and there are no separators between grids. Every output position carries the
coordinates of the cell it predicts **and the step it belongs to**. All 4 × 36 digits are supervised, with no EOS
target. Initialization is identical to Aligned AR.

| run | step embedding |
|---|---|
| `ar_trace_slotstep` (main) | the op-slot embedding of slot s (step s aligned with the operation it executes) |
| `ar_trace_sepstep` (control) | an independent learned step table |

**Evaluation.** Free-running generation of the whole trace; only the final grid is scored. This uses a smaller
sampled test set than before (the trace is 4× longer to decode). The reference is Aligned AR without a trace on the
same setting (notebook 42: single operations 0.00, k = 3 0.00, k = 4 0.06).

**Send back `AlignedAR_trace_SFT_S4_k02_Review.ipynb`.**


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = os.environ.get("DECOMP_MODE", "full")
SEED = 42
RUN_CONDITIONS = ['S4_k02']
RUN_MODELS = ["ar_trace_slotstep", "ar_trace_sepstep"]
STEPS = 20000
SUBDIR = "S7_support_sweep_20k_v1"                # same folder as notebooks 23 / 27 (model subfolders differ)
EVAL_PLAN = {"n_single_grids": 500, "n_seen_per_k": 1000, "n_long_per_k": 4000}   # smaller: 4x longer decode
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
DEVICE = os.environ.get("DECOMP_DEVICE", "auto")
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import longk as LK, longk_exp as LX
print("results ->", LX.family_root(RESULTS_ROOT, MODE, SUBDIR))


In [ ]:
# --- Train (20,000 updates) + sampled LAST-checkpoint evaluation (resumable) ---------------------------------------
status = LX.run_queue(RESULTS_ROOT, MODE, SEED, RUN_CONDITIONS, RUN_MODELS, repo_dir=str(REPO_DIR), device=DEVICE,
                      subdir=SUBDIR, steps=STEPS if MODE == "full" else None, sampled_plan=EVAL_PLAN)
display(status)


In [ ]:
# --- Comparison with Aligned AR without a trace (notebook 42) ---------------------------------------------------------
from decompositionality import shortcut_analysis as SA
models = ["ar_slotpos_digithead_target", "ar_trace_slotstep", "ar_trace_sepstep"]
df = LX.per_k_accuracy(RESULTS_ROOT, MODE, [SEED], ["S4_k02"], SUBDIR, models)
ok = df[df.status == "available"]
print("Exact accuracy of the FINAL grid (rows: model, columns: test k; k = 1 = single operations; trained k = 0, 2):")
display(ok.pivot_table(index="model", columns="k", values="exact_acc").round(4))
rp = SA.analyse(RESULTS_ROOT, MODE, SEED, conditions=["S4_k02"], models=models,
                review_name="AlignedAR_trace_SFT_S4_k02_Review.ipynb", out_name=f"seed_{SEED}_trace_sft_S4_k02")
print("REVIEW FILE TO SEND BACK:", rp)
if IN_COLAB and rp:
    from google.colab import files
    files.download(str(rp))
